# Existing operator chains

## Purpose

This notebook connects every currently implemented operator without claiming
to be a full tree FMM. Four explicit algebraic paths are compared with direct
P2P:

1. P2M $\rightarrow$ M2P
2. P2M $\rightarrow$ M2L $\rightarrow$ L2P
3. P2M(child) $\rightarrow$ M2M(parent) $\rightarrow$ M2P
4. P2M $\rightarrow$ M2L(parent target) $\rightarrow$ L2L(child target) $\rightarrow$ L2P

The result is an interactive analogue of the operator-consistency tests.

## Mathematical perspective

Every path approximates the same potential and field generated by one source
cloud. Differences arise from where a finite Taylor series is centred and
where truncation occurs. Direct P2P supplies the untruncated reference.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import cdfmm

try:
    from example_utils import (
        direct_fields,
        draw_box_3d,
        error_metrics,
        finish_3d_axes,
        local_fields,
        multipole_fields,
        new_3d_figure,
        nodes_at_level,
        plot_coefficients_by_degree,
        random_unit_vectors,
        relative_error,
        set_axes_equal,
        vec3_to_array,
    )
except ModuleNotFoundError:
    # This path is used when the kernel starts in the repository root.
    from examples.notebooks.example_utils import (
        direct_fields,
        draw_box_3d,
        error_metrics,
        finish_3d_axes,
        local_fields,
        multipole_fields,
        new_3d_figure,
        nodes_at_level,
        plot_coefficients_by_degree,
        random_unit_vectors,
        relative_error,
        set_axes_equal,
        vec3_to_array,
    )

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True})

## User parameters

In [ ]:
minimum_order = 2
maximum_order = 6
n_sources = 80
n_targets = 18
random_seed = 42
source_centre = np.zeros(3)
child_source_centre = np.array([0.15, -0.10, 0.10])
parent_target_centre = np.array([4.0, 0.5, -0.25])
child_target_centre = np.array([4.1, 0.4, -0.15])

## Problem setup

In [ ]:
rng = np.random.default_rng(random_seed)
source_positions = child_source_centre + rng.uniform(-0.35, 0.35, size=(n_sources, 3))
dipole_moments = rng.normal(size=(n_sources, 3))
target_positions = child_target_centre + rng.uniform(-0.08, 0.08, size=(n_targets, 3))
reference_fields = direct_fields(target_positions, source_positions, dipole_moments)

print(f"Sources: {n_sources}; targets: {n_targets}")
print(f"Source/target centre separation: {np.linalg.norm(child_target_centre - source_centre):.3f}")

## Evaluate all operator paths

In [ ]:
orders = np.arange(minimum_order, maximum_order + 1)
method_names = [
    "P2M -> M2P",
    "P2M -> M2L -> L2P",
    "P2M(child) -> M2M -> M2P",
    "P2M -> M2L(parent) -> L2L -> L2P",
]
metrics_by_method = {name: [] for name in method_names}

for order in orders:
    parent_multipole = cdfmm.p2m_dipole(
        source_centre,
        source_positions,
        dipole_moments,
        order=order,
    )
    child_multipole = cdfmm.p2m_dipole(
        child_source_centre,
        source_positions,
        dipole_moments,
        order=order,
    )

    path_a = multipole_fields(target_positions, parent_multipole, source_centre, order)

    child_local = cdfmm.m2l(
        parent_multipole,
        source_centre,
        child_target_centre,
        order=order,
    )
    path_b = local_fields(target_positions, child_local, child_target_centre, order)

    translated_parent_multipole = cdfmm.m2m(
        child_multipole,
        child_source_centre,
        source_centre,
        order=order,
    )
    path_c = multipole_fields(
        target_positions,
        translated_parent_multipole,
        source_centre,
        order,
    )

    parent_local = cdfmm.m2l(
        parent_multipole,
        source_centre,
        parent_target_centre,
        order=order,
    )
    translated_child_local = cdfmm.l2l(
        parent_local,
        parent_target_centre,
        child_target_centre,
        order=order,
    )
    path_d = local_fields(
        target_positions,
        translated_child_local,
        child_target_centre,
        order,
    )

    for name, fields in zip(method_names, [path_a, path_b, path_c, path_d]):
        metrics_by_method[name].append(error_metrics(fields, reference_fields))

## Diagnostics table

In [ ]:
print(f"Metrics at p={maximum_order}")
print(" method                                      mean          RMS          maximum")
print("--------------------------------------------------------------------------------")
for method_name in method_names:
    metrics = metrics_by_method[method_name][-1]
    print(
        f" {method_name:43s} "
        f"{metrics['mean']:11.4e}  {metrics['rms']:11.4e}  {metrics['maximum']:11.4e}"
    )

## Convergence visualisation

In [ ]:
figure, axes = plt.subplots(figsize=(9, 5.5))
for method_name in method_names:
    rms_errors = [metrics["rms"] for metrics in metrics_by_method[method_name]]
    axes.semilogy(orders, rms_errors, marker="o", label=method_name)

axes.set_xlabel("Expansion order p")
axes.set_ylabel("RMS relative field error")
axes.set_title("Convergence of current cdfmm operator chains")
axes.legend(fontsize=8)
figure.tight_layout()

## What to observe

All four routes converge toward the same direct field. The precise curves need
not coincide because their centres and truncation sequences differ. These are
manual operator compositions for education and validation—not an upward pass,
downward pass, interaction traversal, or end-to-end tree FMM.